# n-pulse — YOLO26n training on Colab (T4 GPU)

**Before running:** *Runtime > Change runtime type > T4 GPU*. On Google Drive you need:
- `MyDrive/npulse/training_code.zip` (from `bash training/colab/make_code_zip.sh`, re-made after each code change)
- `MyDrive/npulse/data/<dataset>.zip` (from `bash training/colab/make_data_zip.sh <dataset>`, uploaded once)

Runs are written to `MyDrive/npulse/runs/<RUN_NAME>/`. Numbers shown here are for monitoring only:
the reported metrics come from `eval.py` run **locally** (see `training/README.md`).

## 1. Parameters
One run per notebook execution. `EXTRA_ARGS` is passed as-is to `train.py` (e.g. `"--fraction 0.1"`).

In [ ]:
RUN_NAME   = "iter1"
DATA_ZIPS  = ["openimages4.zip"]   # in MyDrive/npulse/data/, unzipped into training/datasets/
EPOCHS     = 30
IMGSZ      = 416
BATCH      = 32
PATIENCE   = 10
EXTRA_ARGS = ""

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/npulse"
for f in ["training_code.zip"] + [f"data/{z}" for z in DATA_ZIPS]:
    print(("ok      " if os.path.exists(f"{DRIVE}/{f}") else "MISSING ") + f"{DRIVE}/{f}")

## 3. Code: unzip `training_code.zip` into `/content/ARM-SOFT-Grasping/`
The zip is copied to the local disk first (reading many small files from Drive is slow).

In [ ]:
import pathlib, shutil, zipfile
REPO = pathlib.Path("/content/ARM-SOFT-Grasping")
REPO.mkdir(parents=True, exist_ok=True)
shutil.copy(f"{DRIVE}/training_code.zip", "/content/training_code.zip")
zipfile.ZipFile("/content/training_code.zip").extractall(REPO)
print(sorted(p.name for p in (REPO / "training").iterdir()))

*Optional, instead of step 3, once the branch is pushed:* get the code with git.

In [ ]:
# !git clone --depth 1 --branch multiclass-detection https://github.com/<owner>/ARM-SOFT-Grasping.git /content/ARM-SOFT-Grasping
# import pathlib; REPO = pathlib.Path("/content/ARM-SOFT-Grasping")

## 4. Data: copy each zip to the local disk, unzip into `training/datasets/`
Training reads images from the local disk, never from Drive.

In [ ]:
DATASETS = REPO / "training" / "datasets"
DATASETS.mkdir(parents=True, exist_ok=True)
for z in DATA_ZIPS:
    local = f"/content/{z}"
    if not os.path.exists(local):
        shutil.copy(f"{DRIVE}/data/{z}", local)
    zipfile.ZipFile(local).extractall(DATASETS)
for d in sorted(DATASETS.iterdir()):
    print(d.name, {s: len(list((d / "images" / s).glob("*"))) for s in ("train", "val", "test")})

## 5. Install Ultralytics (same version as locally) and check the GPU

In [ ]:
!pip install -q ultralytics==8.4.163
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > T4 GPU")

## 6. Train
`training/runs/detect` is a link to `MyDrive/npulse/runs`, so Ultralytics writes `last.pt` / `best.pt`
and `results.csv` straight to Drive after every epoch: a disconnected session loses at most one epoch.

In [ ]:
RUNS_DRIVE = pathlib.Path(DRIVE) / "runs"
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
detect = REPO / "training" / "runs" / "detect"
detect.parent.mkdir(parents=True, exist_ok=True)
if not detect.is_symlink():
    shutil.rmtree(detect, ignore_errors=True)
    detect.symlink_to(RUNS_DRIVE, target_is_directory=True)
if (RUNS_DRIVE / RUN_NAME).exists():
    print(f"WARNING: {RUNS_DRIVE / RUN_NAME} exists -> Ultralytics will create a new name; "
          "change RUN_NAME, pass --exist-ok in EXTRA_ARGS, or use the resume cell below.")

%cd {REPO}
!python training/train.py --device 0 --name {RUN_NAME} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --patience {PATIENCE} {EXTRA_ARGS}

*Optional, after a disconnection:* re-run steps 1–5, then resume from the last checkpoint on Drive.

In [ ]:
# from ultralytics import YOLO
# YOLO(f"{RUNS_DRIVE}/{RUN_NAME}/weights/last.pt").train(resume=True)

## 7. Training curves and final val metrics (monitoring only)

In [ ]:
import pandas as pd
from IPython.display import Image, display
run = RUNS_DRIVE / RUN_NAME
display(Image(filename=str(run / "results.png"), width=1000))
df = pd.read_csv(run / "results.csv")
df.columns = [c.strip() for c in df.columns]
cols = ["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
print("last epochs:"); print(df[cols].tail(3).to_string(index=False))
print("\nbest val mAP50 epoch:"); print(df.loc[[df["metrics/mAP50(B)"].idxmax()], cols].to_string(index=False))
print(f"\nNext: download {run} into training/runs/detect/{RUN_NAME}/ and run eval.py locally.")